# PDF Chunk Search — Production-Grade Rebuild

A hardened rewrite of `search_engine.py`: same feature set (chunk PDFs → store → keyword search
over an HTTP API), rebuilt to standards you would ship, with an end-to-end test suite that runs
inside this notebook.

## What the original does

```python
@app.post("/search_chunks")
def search_chunks(search_string):
    search_list = search_string.split()
    files = glob.glob(os.path.join('Day_1','csv_files', "*.csv"))
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    matches_df = df[df['chunk'].str.contains('|'.join(search_list), case=False, na=False)]
    return matches_df.to_json()
```

## Defects this notebook fixes

| # | Defect in `search_engine.py` | Impact | Fix here |
|---|------------------------------|--------|----------|
| 1 | User input interpolated straight into a regex (`'\|'.join(search_list)`) | A query of `(` raises `re.error` → HTTP 500; `(.*){10}...` is a ReDoS vector | `re.escape` every term |
| 2 | `pdf_file_path` used unchecked | Path traversal — any file the process can read | Resolve and confine to an allowlisted root |
| 3 | Output CSV name derived from the input filename | Traversal on write / silent overwrite | Slugify + confine |
| 4 | `pd.concat` over an empty glob | `ValueError` → HTTP 500 on a fresh install | Explicit empty-corpus branch |
| 5 | Chunking slices by character count | Splits words and UTF-8 graphemes mid-token, so terms vanish | Word-aware windows with overlap |
| 6 | No overlap between chunks | Phrases straddling a boundary are unfindable | Configurable overlap |
| 7 | `num_chunks` unvalidated | `0` → `ZeroDivisionError`; huge values → memory blowup | Pydantic bounds |
| 8 | Reads every CSV on every query | O(corpus) per request | Cached corpus, invalidated on write |
| 9 | Untyped `df.to_json()` | No schema, no pagination, unbounded payload | Typed response models + pagination |
| 10 | No size limit on input PDFs | Memory exhaustion | `max_pdf_bytes` guard |
| 11 | Bare `except`-free handlers | Stack traces leak to clients | Typed exceptions → mapped HTTP codes |
| 12 | No logging, no request IDs | Undebuggable in production | Structured logging |

## Architecture

```
                    ┌─────────────────────────────────┐
   POST /documents  │  api  (FastAPI, thin transport) │  GET /search
        ───────────▶│  validation · errors · logging  │◀───────────
                    └────────────────┬────────────────┘
                                     │
              ┌──────────────────────┼──────────────────────┐
              ▼                      ▼                      ▼
      ┌───────────────┐     ┌────────────────┐     ┌───────────────┐
      │  extraction   │     │   ChunkStore   │     │    search     │
      │ PDF → text    │────▶│  persist/load  │────▶│ rank + page   │
      │ text → chunks │     │  cache         │     │               │
      └───────────────┘     └────────────────┘     └───────────────┘
```

Each layer is independently testable; the API layer holds no business logic. The app is built by a
factory (`build_app(settings)`) so tests get an isolated temp-dir instance instead of mutating the
real corpus.

## 1. Imports and logging

Everything below runs on the packages already pinned in `Week1/requirements.txt` — no new
dependencies. The end-to-end tests drive a **real uvicorn server** over HTTP using `requests`,
which is a stronger guarantee than an in-process test client and avoids needing `httpx`/`pytest`.

In [1]:
from __future__ import annotations

import glob
import io
import json
import logging
import os
import re
import shutil
import socket
import tempfile
import threading
import time
import traceback
import unicodedata
import uuid
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator, Literal, Sequence

import pandas as pd
import requests
import uvicorn
from fastapi import Depends, FastAPI, Query, Request, status
from fastapi.responses import JSONResponse
from PyPDF2 import PdfReader
from pydantic import BaseModel, Field, field_validator

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
# Uvicorn's access log would flood the notebook during the E2E suite.
logging.getLogger("uvicorn.access").setLevel(logging.WARNING)
logging.getLogger("uvicorn.error").setLevel(logging.WARNING)

log = logging.getLogger("search_engine")
log.info("environment ready — pandas %s", pd.__version__)

08:27:59 | INFO    | search_engine | environment ready — pandas 2.2.3


## 2. Configuration

Settings are an immutable dataclass loaded from environment variables with validated defaults.
Immutability matters: config that can be mutated at runtime is config you cannot reason about in
a post-mortem. `pydantic-settings` isn't in this week's lockfile, so this is hand-rolled — the
shape is the same.

`pdf_root` is the security boundary for ingestion: any path outside it is rejected before a file
handle is ever opened.

In [2]:
class ConfigError(RuntimeError):
    """Raised when settings are internally inconsistent."""


@dataclass(frozen=True, slots=True)
class Settings:
    """Immutable application configuration."""

    data_dir: Path            # where chunk CSVs are written
    pdf_root: Path            # ingestion is confined to this subtree
    chunk_size_words: int = 120
    overlap_words: int = 20
    max_pdf_bytes: int = 50 * 1024 * 1024
    max_pages: int = 2_000
    max_query_terms: int = 32
    default_page_size: int = 20
    max_page_size: int = 200

    def __post_init__(self) -> None:
        if self.chunk_size_words < 1:
            raise ConfigError("chunk_size_words must be >= 1")
        if not 0 <= self.overlap_words < self.chunk_size_words:
            raise ConfigError(
                f"overlap_words must satisfy 0 <= overlap < chunk_size "
                f"(got overlap={self.overlap_words}, chunk_size={self.chunk_size_words})"
            )
        if self.max_pdf_bytes < 1:
            raise ConfigError("max_pdf_bytes must be >= 1")
        if self.default_page_size > self.max_page_size:
            raise ConfigError("default_page_size cannot exceed max_page_size")

    @classmethod
    def from_env(cls, **overrides: Any) -> "Settings":
        base = Path(os.environ.get("SEARCH_DATA_DIR", Path.cwd() / "csv_files"))
        values: dict[str, Any] = {
            "data_dir": Path(overrides.pop("data_dir", base)),
            "pdf_root": Path(overrides.pop("pdf_root", os.environ.get("SEARCH_PDF_ROOT", Path.cwd()))),
            "chunk_size_words": int(os.environ.get("SEARCH_CHUNK_SIZE_WORDS", 120)),
            "overlap_words": int(os.environ.get("SEARCH_OVERLAP_WORDS", 20)),
        }
        values.update(overrides)
        return cls(**values)

    def ensure_dirs(self) -> "Settings":
        self.data_dir.mkdir(parents=True, exist_ok=True)
        return self


demo_settings = Settings(
    data_dir=Path.cwd() / "csv_files",
    pdf_root=Path.cwd(),
).ensure_dirs()

print(demo_settings)

Settings(data_dir=PosixPath('/Users/rajuboddula/Downloads/Gen AI - Outskill/Projects/GenAIEngineering-Cohort1/Week1/Day_1/csv_files'), pdf_root=PosixPath('/Users/rajuboddula/Downloads/Gen AI - Outskill/Projects/GenAIEngineering-Cohort1/Week1/Day_1'), chunk_size_words=120, overlap_words=20, max_pdf_bytes=52428800, max_pages=2000, max_query_terms=32, default_page_size=20, max_page_size=200)


### Guardrail: configuration is validated, not trusted

An overlap greater than the chunk size would make the sliding window step backwards and loop
forever. Better to fail at construction than to hang a worker in production.

In [3]:
for bad in (
    dict(chunk_size_words=10, overlap_words=10),   # step == 0 → infinite loop
    dict(chunk_size_words=10, overlap_words=99),   # step < 0  → infinite loop
    dict(chunk_size_words=0),                      # empty windows
):
    try:
        Settings(data_dir=Path("/tmp/x"), pdf_root=Path("/tmp"), **bad)
    except ConfigError as exc:
        print(f"rejected {bad}\n    → {exc}\n")

rejected {'chunk_size_words': 10, 'overlap_words': 10}
    → overlap_words must satisfy 0 <= overlap < chunk_size (got overlap=10, chunk_size=10)

rejected {'chunk_size_words': 10, 'overlap_words': 99}
    → overlap_words must satisfy 0 <= overlap < chunk_size (got overlap=99, chunk_size=10)

rejected {'chunk_size_words': 0}
    → chunk_size_words must be >= 1



## 3. Domain errors

A single exception hierarchy that the API layer maps to status codes. Business logic raises
domain errors and never imports `HTTPException` — that keeps the core reusable from a CLI, a
worker, or a test without dragging FastAPI along.

In [4]:
class SearchEngineError(Exception):
    """Base class for all domain errors. Carries the HTTP status it maps to."""

    status_code: int = status.HTTP_500_INTERNAL_SERVER_ERROR
    code: str = "internal_error"


class DocumentNotFound(SearchEngineError):
    status_code = status.HTTP_404_NOT_FOUND
    code = "document_not_found"


class UnsafePath(SearchEngineError):
    """Requested path escapes the configured root."""

    status_code = status.HTTP_403_FORBIDDEN
    code = "unsafe_path"


class DocumentTooLarge(SearchEngineError):
    status_code = status.HTTP_413_REQUEST_ENTITY_TOO_LARGE
    code = "document_too_large"


class UnreadableDocument(SearchEngineError):
    """File exists but is not a parseable PDF, or contains no extractable text."""

    status_code = status.HTTP_422_UNPROCESSABLE_ENTITY
    code = "unreadable_document"


class InvalidQuery(SearchEngineError):
    status_code = status.HTTP_400_BAD_REQUEST
    code = "invalid_query"

## 4. Safe path resolution

Fix for defects **#2** and **#3**. Two distinct jobs:

- `resolve_within` — proves a *read* path lives under an allowlisted root. `Path.resolve()`
  collapses `..` and follows symlinks first, so `../../etc/passwd` and a symlink pointing outside
  the root are both caught. Checking the string *before* resolution is the classic bypass.
- `safe_stem` — derives a *write* filename from untrusted input by keeping only known-good
  characters, rather than trying to blocklist bad ones.

In [5]:
_SLUG_STRIP = re.compile(r"[^a-z0-9]+")


def resolve_within(root: Path, candidate: Path | str) -> Path:
    """Resolve `candidate` and assert it lives under `root`.

    Raises:
        UnsafePath: if the resolved path escapes `root`.
    """
    root_resolved = Path(root).resolve()
    target = Path(candidate)
    if not target.is_absolute():
        target = root_resolved / target
    target = target.resolve()
    # Compare resolved paths, never raw strings: `..` and symlinks are gone by now.
    if target != root_resolved and root_resolved not in target.parents:
        raise UnsafePath(f"path {candidate!r} resolves outside the permitted root")
    return target


def safe_stem(filename: str, *, max_len: int = 80) -> str:
    """Turn an arbitrary filename into a filesystem-safe slug (allowlist approach)."""
    stem = Path(filename).stem
    # Fold accents so "Résumé.pdf" and "Resume.pdf" don't collide unpredictably.
    stem = unicodedata.normalize("NFKD", stem).encode("ascii", "ignore").decode()
    slug = _SLUG_STRIP.sub("-", stem.lower()).strip("-")
    return (slug[:max_len] or "document")


_root = Path(tempfile.mkdtemp())
(_root / "ok.pdf").write_bytes(b"%PDF-1.4")
print("inside root  :", resolve_within(_root, "ok.pdf").name)
for attack in ("../../../../etc/passwd", "/etc/passwd"):
    try:
        resolve_within(_root, attack)
    except UnsafePath as exc:
        print(f"blocked      : {attack!r} → {exc}")
print("slug         :", safe_stem("../../etc/pa$$wd.pdf"), "|", safe_stem("Résumé (final) v2.pdf"))
shutil.rmtree(_root)

inside root  : ok.pdf
blocked      : '../../../../etc/passwd' → path '../../../../etc/passwd' resolves outside the permitted root
blocked      : '/etc/passwd' → path '/etc/passwd' resolves outside the permitted root
slug         : pa-wd | resume-final-v2


## 5. Text extraction and chunking

Fix for defects **#5**, **#6**, **#7**.

The original computed `chunk_size = len(text) // num_chunks` and sliced by character offset. Two
consequences: a word straddling a boundary is destroyed (`"invoice"` → `"inv"` + `"oice"`, so
neither chunk matches a search for `invoice`), and chunk length varies wildly with page length.

Here chunks are **fixed-size word windows with overlap**. Overlap is what makes phrases spanning a
boundary retrievable — without it, a chunk break in the middle of "annual recurring revenue"
makes that phrase unfindable by any single chunk.

```
words:  w1 w2 w3 w4 w5 w6 w7 w8 w9 ...
chunk1: [w1 .. w5]
chunk2:       [w4 .. w8]      ← overlap of 2 preserves cross-boundary phrases
chunk3:             [w7 ..]
step = chunk_size - overlap
```

In [6]:
_WHITESPACE = re.compile(r"\s+")


def normalise_text(raw: str) -> str:
    """Collapse whitespace and strip control characters from extracted PDF text."""
    cleaned = "".join(ch for ch in raw if ch == "\n" or ch == "\t" or unicodedata.category(ch)[0] != "C")
    return _WHITESPACE.sub(" ", cleaned).strip()


def window_words(words: Sequence[str], size: int, overlap: int) -> Iterator[list[str]]:
    """Yield overlapping fixed-size windows. Guaranteed to terminate: step >= 1."""
    step = size - overlap
    if step < 1:                      # defence in depth; Settings already enforces this
        raise ConfigError("overlap must be smaller than chunk size")
    if not words:
        return
    start = 0
    while start < len(words):
        yield list(words[start : start + size])
        if start + size >= len(words):
            break                     # final window reached the end; don't emit a tail duplicate
        start += step


@dataclass(frozen=True, slots=True)
class Chunk:
    """One retrievable unit of text, with provenance back to its source page."""

    document_id: str
    filename: str
    page_number: int
    chunk_number: int
    text: str

    @property
    def chunk_id(self) -> str:
        return f"{self.document_id}:{self.page_number}:{self.chunk_number}"


def extract_pages(pdf_path: Path, *, max_pages: int) -> list[str]:
    """Extract normalised text per page. Blank pages are preserved as '' to keep numbering true."""
    try:
        reader = PdfReader(str(pdf_path))
        pages = reader.pages[:max_pages]
        return [normalise_text(page.extract_text() or "") for page in pages]
    except UnreadableDocument:
        raise
    except Exception as exc:  # PyPDF2 raises a wide variety of parse errors
        raise UnreadableDocument(f"could not parse {pdf_path.name}: {exc}") from exc


def chunk_document(
    pdf_path: Path,
    *,
    document_id: str,
    chunk_size_words: int,
    overlap_words: int,
    max_pages: int,
) -> list[Chunk]:
    """Extract and chunk a PDF. Page numbers are 1-based and survive blank pages."""
    pages = extract_pages(pdf_path, max_pages=max_pages)
    chunks: list[Chunk] = []
    for page_number, page_text in enumerate(pages, start=1):
        words = page_text.split()
        for chunk_number, window in enumerate(window_words(words, chunk_size_words, overlap_words), start=1):
            chunks.append(
                Chunk(
                    document_id=document_id,
                    filename=pdf_path.name,
                    page_number=page_number,
                    chunk_number=chunk_number,
                    text=" ".join(window),
                )
            )
    if not chunks:
        # A scanned/image-only PDF parses fine but yields nothing — that is a client-visible
        # condition (needs OCR), not a server fault.
        raise UnreadableDocument(
            f"{pdf_path.name} contains no extractable text (is it a scanned image?)"
        )
    return chunks


demo_words = "the quick brown fox jumps over the lazy dog near the river bank".split()
for i, w in enumerate(window_words(demo_words, size=5, overlap=2), 1):
    print(f"chunk {i}: {' '.join(w)}")

chunk 1: the quick brown fox jumps
chunk 2: fox jumps over the lazy
chunk 3: the lazy dog near the
chunk 4: near the river bank


### Why overlap is not optional

Below, the phrase *"lazy dog"* straddles a window boundary. With `overlap=0` no single chunk
contains it and the phrase is unretrievable; with overlap it survives intact.

In [7]:
phrase = "lazy dog"
for overlap in (0, 2):
    chunks = [" ".join(w) for w in window_words(demo_words, size=5, overlap=overlap)]
    hit = any(phrase in c for c in chunks)
    print(f"overlap={overlap}: {'FOUND' if hit else 'LOST '} {phrase!r}  chunks={chunks}")

overlap=0: FOUND 'lazy dog'  chunks=['the quick brown fox jumps', 'over the lazy dog near', 'the river bank']
overlap=2: FOUND 'lazy dog'  chunks=['the quick brown fox jumps', 'fox jumps over the lazy', 'the lazy dog near the', 'near the river bank']


## 6. Storage layer

`ChunkStore` owns all filesystem access. Two properties worth calling out:

**Atomic writes.** Chunks are written to a temp file and then `os.replace`d into position.
`os.replace` is atomic on POSIX and Windows, so a crash mid-write can never leave a reader
observing a half-written CSV. The naive `df.to_csv(final_path)` in the original can.

**Cache invalidation (defect #8).** The original re-read and re-concatenated every CSV on every
query. Here the corpus is cached in memory and the cache is dropped on write. The cache is
guarded by a lock because uvicorn serves requests from a thread pool.

In [8]:
CHUNK_COLUMNS = ["document_id", "filename", "page_number", "chunk_number", "text"]


class ChunkStore:
    """Persists chunks as one CSV per document and serves a cached in-memory corpus."""

    def __init__(self, settings: Settings) -> None:
        self._settings = settings
        self._lock = threading.RLock()
        self._cache: pd.DataFrame | None = None
        settings.data_dir.mkdir(parents=True, exist_ok=True)

    def _path_for(self, document_id: str) -> Path:
        return self._settings.data_dir / f"{document_id}.csv"

    def write(self, document_id: str, chunks: Sequence[Chunk]) -> Path:
        """Atomically persist a document's chunks and invalidate the corpus cache."""
        frame = pd.DataFrame(
            [
                {
                    "document_id": c.document_id,
                    "filename": c.filename,
                    "page_number": c.page_number,
                    "chunk_number": c.chunk_number,
                    "text": c.text,
                }
                for c in chunks
            ],
            columns=CHUNK_COLUMNS,
        )
        target = self._path_for(document_id)
        with self._lock:
            fd, tmp_name = tempfile.mkstemp(dir=self._settings.data_dir, suffix=".tmp")
            os.close(fd)
            tmp = Path(tmp_name)
            try:
                frame.to_csv(tmp, index=False)
                os.replace(tmp, target)   # atomic: readers see old or new, never partial
            finally:
                tmp.unlink(missing_ok=True)
            self._cache = None
        log.info("stored %d chunks for document_id=%s", len(chunks), document_id)
        return target

    def delete(self, document_id: str) -> None:
        with self._lock:
            path = self._path_for(document_id)
            if not path.exists():
                raise DocumentNotFound(f"no document with id {document_id!r}")
            path.unlink()
            self._cache = None

    def corpus(self) -> pd.DataFrame:
        """Return all chunks as one DataFrame. Empty (but correctly typed) when no documents."""
        with self._lock:
            if self._cache is not None:
                return self._cache
            files = sorted(self._settings.data_dir.glob("*.csv"))
            if not files:
                # Defect #4: pd.concat([]) raises ValueError. Return a typed empty frame.
                self._cache = pd.DataFrame(columns=CHUNK_COLUMNS)
                return self._cache
            frames = []
            for path in files:
                try:
                    frames.append(pd.read_csv(path, dtype={"text": "string"}))
                except (pd.errors.EmptyDataError, pd.errors.ParserError):
                    log.warning("skipping unreadable corpus file %s", path.name)
            self._cache = (
                pd.concat(frames, ignore_index=True) if frames
                else pd.DataFrame(columns=CHUNK_COLUMNS)
            )
            return self._cache

    def documents(self) -> list[dict[str, Any]]:
        corpus = self.corpus()
        if corpus.empty:
            return []
        grouped = corpus.groupby(["document_id", "filename"], as_index=False).agg(
            chunk_count=("text", "size"), page_count=("page_number", "nunique")
        )
        return grouped.to_dict("records")


_tmp = Path(tempfile.mkdtemp())
_store = ChunkStore(Settings(data_dir=_tmp, pdf_root=_tmp))
print("empty corpus is safe :", _store.corpus().shape, list(_store.corpus().columns))
print("documents            :", _store.documents())
shutil.rmtree(_tmp)

empty corpus is safe : (0, 5) ['document_id', 'filename', 'page_number', 'chunk_number', 'text']
documents            : []


## 7. Search

Fix for defects **#1** and **#9**.

The injection: `'|'.join(search_list)` treats user input as a regex. A query of `(` is an
unbalanced group → `re.error` → HTTP 500. A crafted query like `(a+)+$` is catastrophic
backtracking (ReDoS) that pins a CPU core. `re.escape` on every term closes both.

Matching is on **word boundaries**, so searching `art` does not match `start` — substring
matching produces noise that users read as broken search.

Ranking is a simple, explainable score: number of distinct query terms present, then total
occurrences. No TF-IDF here; the point is that ranking is deliberate and testable rather than
"whatever order the files globbed in".

In [9]:
SearchMode = Literal["any", "all"]


def compile_terms(query: str, *, max_terms: int) -> list[tuple[str, re.Pattern[str]]]:
    """Compile each whitespace-separated term into a case-insensitive word-boundary pattern.

    Returns (original_term, pattern) pairs — the raw term is kept so results can report which
    terms matched without trying to un-escape a compiled pattern.
    """
    terms = [t for t in query.split() if t.strip()]
    if not terms:
        raise InvalidQuery("query must contain at least one search term")
    if len(terms) > max_terms:
        raise InvalidQuery(f"query has {len(terms)} terms; the maximum is {max_terms}")
    # re.escape is the fix for defect #1 — user input becomes a literal, never an operator.
    return [(t, re.compile(rf"\b{re.escape(t)}\b", re.IGNORECASE)) for t in terms]


@dataclass(frozen=True, slots=True)
class SearchHit:
    chunk_id: str
    document_id: str
    filename: str
    page_number: int
    chunk_number: int
    text: str
    score: int
    matched_terms: list[str] = field(default_factory=list)


def search_corpus(
    corpus: pd.DataFrame,
    query: str,
    *,
    mode: SearchMode = "any",
    max_terms: int = 32,
) -> list[SearchHit]:
    """Rank chunks by distinct-term coverage, then by total occurrences."""
    compiled = compile_terms(query, max_terms=max_terms)
    if corpus.empty:
        return []

    hits: list[SearchHit] = []
    for row in corpus.itertuples(index=False):
        text = str(row.text)
        matched, occurrences = [], 0
        for term, pattern in compiled:
            found = pattern.findall(text)
            if found:
                matched.append(term)
                occurrences += len(found)
        if not matched:
            continue
        if mode == "all" and len(matched) != len(compiled):
            continue
        hits.append(
            SearchHit(
                chunk_id=f"{row.document_id}:{row.page_number}:{row.chunk_number}",
                document_id=str(row.document_id),
                filename=str(row.filename),
                page_number=int(row.page_number),
                chunk_number=int(row.chunk_number),
                text=text,
                score=len(matched) * 1000 + occurrences,   # coverage dominates frequency
                matched_terms=matched,
            )
        )
    # Stable, deterministic ordering — identical queries always return identical pages.
    hits.sort(key=lambda h: (-h.score, h.filename, h.page_number, h.chunk_number))
    return hits


sample = pd.DataFrame(
    [
        {"document_id": "d1", "filename": "a.pdf", "page_number": 1, "chunk_number": 1,
         "text": "The quarterly revenue report shows revenue growth."},
        {"document_id": "d1", "filename": "a.pdf", "page_number": 1, "chunk_number": 2,
         "text": "Growth in the start-up sector was modest."},
        {"document_id": "d2", "filename": "b.pdf", "page_number": 3, "chunk_number": 1,
         "text": "Revenue and growth both exceeded the forecast."},
    ]
)

print("mode=any :", [(h.filename, h.score) for h in search_corpus(sample, "revenue growth", mode="any")])
print("mode=all :", [(h.filename, h.score) for h in search_corpus(sample, "revenue growth", mode="all")])
print("word-boundary — 'art' does not match 'start':",
      search_corpus(sample, "art") == [])

mode=any : [('a.pdf', 2003), ('b.pdf', 2002), ('a.pdf', 1001)]
mode=all : [('a.pdf', 2003), ('b.pdf', 2002)]
word-boundary — 'art' does not match 'start': True


### The injection, demonstrated

Side by side: the original's matching versus the hardened version, on inputs a user can type.

In [10]:
def original_search(df: pd.DataFrame, search_string: str):
    """Verbatim matching logic from search_engine.py, for comparison."""
    search_list = search_string.split()
    return df[df["text"].str.contains("|".join(search_list), case=False, na=False)]


for hostile in ["(", "revenue)", "a{2,3"]:
    try:
        original_search(sample, hostile)
        outcome = "no error"
    except re.error as exc:
        outcome = f"re.error: {exc}"
    print(f"query {hostile!r:14} original → {outcome}")

    try:
        hits = search_corpus(sample, hostile)
        print(f"{'':21}hardened → {len(hits)} hits, no error")
    except SearchEngineError as exc:
        print(f"{'':21}hardened → {type(exc).__name__} (clean 4xx)")

query '('            original → re.error: missing ), unterminated subpattern at position 0
                     hardened → 0 hits, no error
query 'revenue)'     original → re.error: unbalanced parenthesis at position 7
                     hardened → 0 hits, no error
query 'a{2,3'        original → no error
                     hardened → 0 hits, no error


## 8. Service layer

Orchestrates extraction → storage → search. This class is the actual application; the FastAPI
layer below is a thin adapter over it. You could put a CLI or a queue consumer on top of the
same object without changing a line.

`document_id` is a deterministic hash of the resolved path, so re-ingesting the same file
updates its chunks in place rather than duplicating them.

In [11]:
@dataclass(frozen=True, slots=True)
class IngestResult:
    document_id: str
    filename: str
    page_count: int
    chunk_count: int


class SearchService:
    def __init__(self, settings: Settings, store: ChunkStore | None = None) -> None:
        self.settings = settings
        self.store = store or ChunkStore(settings)

    @staticmethod
    def _document_id(path: Path) -> str:
        # Deterministic: same path → same id → re-ingest overwrites instead of duplicating.
        return f"{safe_stem(path.name)}-{uuid.uuid5(uuid.NAMESPACE_URL, str(path)).hex[:8]}"

    def ingest(self, pdf_path: str, *, chunk_size_words: int | None = None,
               overlap_words: int | None = None) -> IngestResult:
        resolved = resolve_within(self.settings.pdf_root, pdf_path)   # defect #2
        if not resolved.is_file():
            raise DocumentNotFound(f"no such file: {pdf_path}")

        size = resolved.stat().st_size
        if size > self.settings.max_pdf_bytes:                        # defect #10
            raise DocumentTooLarge(
                f"{resolved.name} is {size} bytes; the limit is {self.settings.max_pdf_bytes}"
            )
        if resolved.suffix.lower() != ".pdf":
            raise UnreadableDocument(f"{resolved.name} is not a .pdf file")

        size_words = chunk_size_words or self.settings.chunk_size_words
        overlap = self.settings.overlap_words if overlap_words is None else overlap_words
        if overlap >= size_words:
            raise InvalidQuery("overlap_words must be smaller than chunk_size_words")

        document_id = self._document_id(resolved)
        chunks = chunk_document(
            resolved,
            document_id=document_id,
            chunk_size_words=size_words,
            overlap_words=overlap,
            max_pages=self.settings.max_pages,
        )
        self.store.write(document_id, chunks)
        return IngestResult(
            document_id=document_id,
            filename=resolved.name,
            page_count=len({c.page_number for c in chunks}),
            chunk_count=len(chunks),
        )

    def search(self, query: str, *, mode: SearchMode = "any",
               limit: int = 20, offset: int = 0) -> tuple[list[SearchHit], int]:
        """Return one page of hits plus the total match count (defect #9)."""
        hits = search_corpus(
            self.store.corpus(), query, mode=mode, max_terms=self.settings.max_query_terms
        )
        return hits[offset : offset + limit], len(hits)

    def documents(self) -> list[dict[str, Any]]:
        return self.store.documents()

    def delete(self, document_id: str) -> None:
        self.store.delete(document_id)

## 9. API schemas

Explicit request/response models replace the original's untyped `df.to_json()`. These give you
validation at the boundary, a generated OpenAPI spec, and a contract that tests can assert
against.

In [12]:
class IngestRequest(BaseModel):
    pdf_path: str = Field(..., min_length=1, description="Path to a PDF, relative to the configured root")
    chunk_size_words: int | None = Field(None, ge=10, le=2000)
    overlap_words: int | None = Field(None, ge=0, le=500)

    @field_validator("pdf_path")
    @classmethod
    def _not_blank(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("pdf_path must not be blank")
        return v.strip()


class IngestResponse(BaseModel):
    document_id: str
    filename: str
    page_count: int
    chunk_count: int


class SearchHitModel(BaseModel):
    chunk_id: str
    document_id: str
    filename: str
    page_number: int
    chunk_number: int
    text: str
    score: int
    matched_terms: list[str]


class SearchResponse(BaseModel):
    query: str
    mode: SearchMode
    total: int
    limit: int
    offset: int
    results: list[SearchHitModel]


class DocumentSummary(BaseModel):
    document_id: str
    filename: str
    chunk_count: int
    page_count: int


class HealthResponse(BaseModel):
    status: Literal["healthy"]
    documents: int
    chunks: int


class ErrorResponse(BaseModel):
    code: str
    detail: str
    request_id: str

## 10. The application factory

`build_app(settings)` rather than a module-level `app = FastAPI()`. This is what makes the app
testable: each test constructs an instance backed by its own temp directory, so tests neither
touch the real corpus nor interfere with each other.

Three things the original lacked:

- **Request IDs** — every response carries `X-Request-ID`, echoed in logs and error bodies, so a
  user-reported failure can be traced to a specific log line.
- **Domain-error handler** — `SearchEngineError` subclasses map to their declared status codes.
- **Catch-all handler** — unexpected exceptions log a full traceback server-side but return a
  generic body, so internal paths and stack frames never reach a client.

In [13]:
def build_app(settings: Settings) -> FastAPI:
    settings.ensure_dirs()
    service = SearchService(settings)

    app = FastAPI(
        title="PDF Chunk Search",
        version="2.0.0",
        summary="Chunk PDFs into retrievable passages and search them by keyword.",
    )
    app.state.service = service

    def get_service() -> SearchService:
        return app.state.service

    @app.middleware("http")
    async def add_request_id(request: Request, call_next):
        request_id = request.headers.get("X-Request-ID", uuid.uuid4().hex[:12])
        request.state.request_id = request_id
        start = time.perf_counter()
        response = await call_next(request)
        elapsed_ms = (time.perf_counter() - start) * 1000
        response.headers["X-Request-ID"] = request_id
        log.info(
            "%s %s → %s (%.1f ms) [%s]",
            request.method, request.url.path, response.status_code, elapsed_ms, request_id,
        )
        return response

    @app.exception_handler(SearchEngineError)
    async def handle_domain_error(request: Request, exc: SearchEngineError):
        request_id = getattr(request.state, "request_id", "-")
        log.warning("%s: %s [%s]", exc.code, exc, request_id)
        return JSONResponse(
            status_code=exc.status_code,
            content=ErrorResponse(code=exc.code, detail=str(exc), request_id=request_id).model_dump(),
        )

    @app.exception_handler(Exception)
    async def handle_unexpected(request: Request, exc: Exception):
        request_id = getattr(request.state, "request_id", "-")
        # Full detail to the log, generic message to the client (defect #11).
        log.error("unhandled error [%s]\n%s", request_id, traceback.format_exc())
        return JSONResponse(
            status_code=500,
            content=ErrorResponse(
                code="internal_error", detail="An unexpected error occurred.", request_id=request_id
            ).model_dump(),
        )

    @app.get("/health", response_model=HealthResponse, tags=["ops"])
    async def health(svc: SearchService = Depends(get_service)) -> HealthResponse:
        corpus = svc.store.corpus()
        return HealthResponse(
            status="healthy",
            documents=int(corpus["document_id"].nunique()) if not corpus.empty else 0,
            chunks=int(len(corpus)),
        )

    @app.post("/documents", response_model=IngestResponse,
              status_code=status.HTTP_201_CREATED, tags=["documents"])
    async def ingest_document(
        payload: IngestRequest, svc: SearchService = Depends(get_service)
    ) -> IngestResponse:
        result = svc.ingest(
            payload.pdf_path,
            chunk_size_words=payload.chunk_size_words,
            overlap_words=payload.overlap_words,
        )
        return IngestResponse(**result.__dict__)

    @app.get("/documents", response_model=list[DocumentSummary], tags=["documents"])
    async def list_documents(svc: SearchService = Depends(get_service)) -> list[DocumentSummary]:
        return [DocumentSummary(**d) for d in svc.documents()]

    @app.delete("/documents/{document_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["documents"])
    async def delete_document(document_id: str, svc: SearchService = Depends(get_service)) -> None:
        svc.delete(document_id)

    @app.get("/search", response_model=SearchResponse, tags=["search"])
    async def search(
        q: str = Query(..., min_length=1, description="Whitespace-separated search terms"),
        mode: SearchMode = Query("any", description="'any' = OR across terms, 'all' = AND"),
        limit: int = Query(settings.default_page_size, ge=1, le=settings.max_page_size),
        offset: int = Query(0, ge=0),
        svc: SearchService = Depends(get_service),
    ) -> SearchResponse:
        hits, total = svc.search(q, mode=mode, limit=limit, offset=offset)
        return SearchResponse(
            query=q, mode=mode, total=total, limit=limit, offset=offset,
            results=[SearchHitModel(**h.__dict__) for h in hits],
        )

    return app


print("app factory ready")

app factory ready


## 11. Test fixtures — building real PDFs with no extra dependencies

`reportlab` isn't in this week's lockfile, so the tests synthesise **valid PDF bytes directly**.
This keeps the suite hermetic: no network, no binary fixtures committed to the repo, and PyPDF2
does the real parsing work under test.

The structure below is the minimum a conforming PDF needs: a catalog, a page tree, a font
resource, one content stream per page, and a cross-reference table with byte offsets.

In [14]:
def _escape_pdf_text(text: str) -> str:
    return text.replace("\\", r"\\").replace("(", r"\(").replace(")", r"\)")


def _wrap(text: str, width: int = 90) -> list[str]:
    words, lines, current = text.split(), [], ""
    for word in words:
        if len(current) + len(word) + 1 > width:
            lines.append(current)
            current = word
        else:
            current = f"{current} {word}".strip()
    if current:
        lines.append(current)
    return lines or [""]


def make_pdf_bytes(pages: Sequence[str]) -> bytes:
    """Build a minimal, valid, text-extractable PDF from page strings."""
    objects: list[bytes] = []
    n_pages = len(pages)
    # Object ids: 1=catalog, 2=page tree, 3=font, then page/content pairs from 4.
    page_ids = [4 + 2 * i for i in range(n_pages)]
    content_ids = [5 + 2 * i for i in range(n_pages)]

    kids = " ".join(f"{pid} 0 R" for pid in page_ids)
    objects.append(b"<< /Type /Catalog /Pages 2 0 R >>")
    objects.append(f"<< /Type /Pages /Kids [{kids}] /Count {n_pages} >>".encode())
    objects.append(b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")

    for page_text, content_id in zip(pages, content_ids):
        objects.append(
            (
                f"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
                f"/Resources << /Font << /F1 3 0 R >> >> /Contents {content_id} 0 R >>"
            ).encode()
        )
        lines = "\n".join(f"({_escape_pdf_text(line)}) Tj T*" for line in _wrap(page_text))
        stream = f"BT\n/F1 11 Tf\n72 720 Td\n14 TL\n{lines}\nET".encode()
        objects.append(b"<< /Length " + str(len(stream)).encode() + b" >>\nstream\n" + stream + b"\nendstream")

    out = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for i, body in enumerate(objects, start=1):
        offsets.append(len(out))
        out += f"{i} 0 obj\n".encode() + body + b"\nendobj\n"

    xref_pos = len(out)
    out += f"xref\n0 {len(objects) + 1}\n".encode()
    out += b"0000000000 65535 f \n"
    for offset in offsets[1:]:
        out += f"{offset:010d} 00000 n \n".encode()
    out += (
        f"trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_pos}\n%%EOF\n"
    ).encode()
    return bytes(out)


# Verify the fixture generator round-trips through PyPDF2 before relying on it.
_probe = make_pdf_bytes(["Hello world from page one.", "Second page mentions revenue."])
_probe_path = Path(tempfile.mkdtemp()) / "probe.pdf"
_probe_path.write_bytes(_probe)
_reader = PdfReader(str(_probe_path))
print(f"fixture generator OK — {len(_reader.pages)} pages")
for i, p in enumerate(_reader.pages, 1):
    print(f"  page {i}: {normalise_text(p.extract_text())!r}")
shutil.rmtree(_probe_path.parent)

fixture generator OK — 2 pages
  page 1: 'Hello world from page one.'
  page 2: 'Second page mentions revenue.'


## 12. Test harness

`pytest` isn't in this week's lockfile either, so here is a ~40-line runner with the pieces that
matter: per-test isolation, assertion helpers with useful failure messages, exception capture,
and a summary that reports a non-zero failure count rather than dying on the first error.

The fixture is a context manager that yields an isolated `Settings` + `SearchService` + sample
PDFs on a fresh temp directory, and cleans up unconditionally.

In [15]:
_TESTS: list[tuple[str, Any]] = []


def test(fn):
    """Register a test function. Name doubles as the report label."""
    _TESTS.append((fn.__name__, fn))
    return fn


def assert_eq(actual, expected, msg=""):
    if actual != expected:
        raise AssertionError(f"{msg}\n  expected: {expected!r}\n  actual:   {actual!r}")


def assert_true(cond, msg=""):
    if not cond:
        raise AssertionError(msg or "expected a truthy value")


def assert_raises(exc_type, fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except exc_type as exc:
        return exc
    except Exception as other:
        raise AssertionError(f"expected {exc_type.__name__}, got {type(other).__name__}: {other}")
    raise AssertionError(f"expected {exc_type.__name__}, but no exception was raised")


SAMPLE_PAGES = {
    "annual_report.pdf": [
        "The annual report opens with a summary of quarterly revenue and steady growth "
        "across every operating region during the financial year.",
        "Risk factors include supply chain disruption and currency exposure. "
        "The board considers revenue concentration the principal risk.",
    ],
    "handbook.pdf": [
        "The employee handbook describes onboarding, the expense policy, and how to "
        "request annual leave through the internal portal.",
    ],
}


@contextmanager
def isolated_service():
    """Yield (settings, service, pdf_dir) backed by a throwaway temp directory."""
    root = Path(tempfile.mkdtemp(prefix="search-test-"))
    try:
        pdf_dir = root / "pdfs"
        pdf_dir.mkdir()
        for name, pages in SAMPLE_PAGES.items():
            (pdf_dir / name).write_bytes(make_pdf_bytes(pages))
        settings = Settings(
            data_dir=root / "csv_files",
            pdf_root=pdf_dir,
            chunk_size_words=15,
            overlap_words=5,
        ).ensure_dirs()
        yield settings, SearchService(settings), pdf_dir
    finally:
        shutil.rmtree(root, ignore_errors=True)


def run_tests(only: str | None = None) -> int:
    selected = [(n, f) for n, f in _TESTS if only is None or only in n]
    passed, failures = 0, []
    print(f"running {len(selected)} tests\n" + "=" * 72)
    for name, fn in selected:
        started = time.perf_counter()
        try:
            fn()
        except Exception:
            failures.append((name, traceback.format_exc()))
            print(f"FAIL  {name}")
        else:
            passed += 1
            print(f"pass  {name}  ({(time.perf_counter() - started) * 1000:.0f} ms)")
    print("=" * 72)
    for name, tb in failures:
        print(f"\n--- {name} ---\n{tb}")
    print(f"\n{passed} passed, {len(failures)} failed, {len(selected)} total")
    return len(failures)


print("harness ready")

harness ready


## 13. Unit tests

One behaviour per test, named so a failure report reads like a sentence. Note the tests are
written against *behaviour a user can observe*, not against internal implementation details —
they would survive a rewrite of the chunker's internals.

In [16]:
# --- chunking ------------------------------------------------------------------

@test
def test_windows_cover_all_words():
    words = [f"w{i}" for i in range(23)]
    covered = {w for window in window_words(words, 10, 3) for w in window}
    assert_eq(covered, set(words), "every word must appear in at least one window")


@test
def test_windows_overlap_by_configured_amount():
    words = [f"w{i}" for i in range(20)]
    windows = list(window_words(words, 8, 3))
    assert_eq(windows[0][-3:], windows[1][:3], "consecutive windows must share `overlap` words")


@test
def test_single_window_when_text_shorter_than_chunk_size():
    assert_eq(len(list(window_words(["a", "b"], 50, 10))), 1)


@test
def test_empty_input_yields_no_windows():
    assert_eq(list(window_words([], 10, 2)), [])


@test
def test_window_terminates_and_does_not_duplicate_tail():
    # Regression guard: an off-by-one in the break condition emits a repeated final window.
    words = [f"w{i}" for i in range(21)]
    windows = list(window_words(words, 10, 5))
    assert_true(windows[-1] != windows[-2], "final window must not duplicate its predecessor")


@test
def test_words_are_never_split_mid_token():
    # The core defect in the original: character slicing destroys words.
    with isolated_service() as (settings, service, pdf_dir):
        chunks = chunk_document(
            pdf_dir / "annual_report.pdf", document_id="d", chunk_size_words=15,
            overlap_words=5, max_pages=100,
        )
        source_words = set()
        for page in extract_pages(pdf_dir / "annual_report.pdf", max_pages=100):
            source_words.update(page.split())
        chunk_words = {w for c in chunks for w in c.text.split()}
        assert_true(chunk_words <= source_words, "chunking invented a token — a word was split")


@test
def test_normalise_text_collapses_whitespace():
    assert_eq(normalise_text("  hello \n\n  world \t "), "hello world")


# --- path safety ---------------------------------------------------------------

@test
def test_traversal_outside_root_is_rejected():
    with isolated_service() as (settings, service, pdf_dir):
        for attack in ("../../../etc/passwd", "/etc/passwd", "../pdfs/../../../etc/hosts"):
            assert_raises(UnsafePath, resolve_within, settings.pdf_root, attack)


@test
def test_symlink_escaping_root_is_rejected():
    with isolated_service() as (settings, service, pdf_dir):
        outside = pdf_dir.parent / "secret.pdf"
        outside.write_bytes(b"%PDF-1.4 secret")
        link = pdf_dir / "innocent.pdf"
        try:
            link.symlink_to(outside)
        except (OSError, NotImplementedError):
            return  # platform without symlink permission; nothing to assert
        # Resolution follows the symlink first, so the escape is caught.
        assert_raises(UnsafePath, resolve_within, settings.pdf_root, "innocent.pdf")


@test
def test_safe_stem_strips_dangerous_characters():
    assert_eq(safe_stem("../../etc/pa$$wd.pdf"), "pa-wd")
    assert_eq(safe_stem("....pdf"), "document")
    assert_true("/" not in safe_stem("a/b/c.pdf"))


# --- search --------------------------------------------------------------------

@test
def test_regex_metacharacters_are_treated_as_literals():
    # Defect #1: these inputs crash the original with re.error.
    df = pd.DataFrame([{"document_id": "d", "filename": "f.pdf", "page_number": 1,
                        "chunk_number": 1, "text": "a literal (paren) and a+b here"}])
    assert_eq(len(search_corpus(df, "(paren)")), 0, "escaped parens should not match bare text")
    assert_true(len(search_corpus(df, "a+b")) == 1, "literal 'a+b' should match")
    for hostile in ["(", ")", "[", "a{2,", "*", "(a+)+$"]:
        search_corpus(df, hostile)  # must not raise


@test
def test_search_matches_whole_words_only():
    df = pd.DataFrame([{"document_id": "d", "filename": "f.pdf", "page_number": 1,
                        "chunk_number": 1, "text": "the startup will start soon"}])
    assert_eq(len(search_corpus(df, "art")), 0, "'art' must not match inside 'start'")
    assert_eq(len(search_corpus(df, "start")), 1)


@test
def test_mode_all_requires_every_term():
    assert_eq(len(search_corpus(sample, "revenue growth", mode="any")), 3)
    assert_eq(len(search_corpus(sample, "revenue growth", mode="all")), 2)


@test
def test_ranking_prefers_broader_term_coverage():
    hits = search_corpus(sample, "revenue growth", mode="any")
    coverage = [len(h.matched_terms) for h in hits]
    assert_eq(coverage, sorted(coverage, reverse=True), "hits must be ordered by coverage")


@test
def test_ranking_is_deterministic():
    a = [h.chunk_id for h in search_corpus(sample, "revenue growth")]
    b = [h.chunk_id for h in search_corpus(sample, "revenue growth")]
    assert_eq(a, b)


@test
def test_blank_query_is_rejected():
    assert_raises(InvalidQuery, search_corpus, sample, "   ")


@test
def test_too_many_terms_is_rejected():
    assert_raises(InvalidQuery, search_corpus, sample, " ".join(["x"] * 40), max_terms=32)


@test
def test_search_over_empty_corpus_returns_empty():
    empty = pd.DataFrame(columns=CHUNK_COLUMNS)
    assert_eq(search_corpus(empty, "anything"), [])


# --- storage -------------------------------------------------------------------

@test
def test_empty_corpus_does_not_raise():
    # Defect #4: pd.concat over an empty glob raises ValueError in the original.
    with isolated_service() as (settings, service, pdf_dir):
        corpus = service.store.corpus()
        assert_true(corpus.empty)
        assert_eq(list(corpus.columns), CHUNK_COLUMNS)


@test
def test_cache_is_invalidated_on_write():
    with isolated_service() as (settings, service, pdf_dir):
        assert_eq(len(service.store.corpus()), 0)
        service.ingest("annual_report.pdf")
        assert_true(len(service.store.corpus()) > 0, "cache must refresh after a write")


@test
def test_reingest_replaces_rather_than_duplicates():
    with isolated_service() as (settings, service, pdf_dir):
        first = service.ingest("annual_report.pdf")
        second = service.ingest("annual_report.pdf")
        assert_eq(first.document_id, second.document_id, "document id must be deterministic")
        assert_eq(len(service.documents()), 1, "re-ingest must not create a second document")


@test
def test_delete_removes_document():
    with isolated_service() as (settings, service, pdf_dir):
        result = service.ingest("handbook.pdf")
        service.delete(result.document_id)
        assert_eq(service.documents(), [])
        assert_raises(DocumentNotFound, service.delete, result.document_id)


@test
def test_corpus_survives_a_corrupt_csv():
    with isolated_service() as (settings, service, pdf_dir):
        service.ingest("handbook.pdf")
        (settings.data_dir / "corrupt.csv").write_text("not,a,valid\ncsv\"\"\"row,,,,\n")
        corpus = service.store.corpus()   # must skip the bad file, not raise
        assert_true(len(corpus) > 0)


# --- service -------------------------------------------------------------------

@test
def test_ingest_reports_accurate_counts():
    with isolated_service() as (settings, service, pdf_dir):
        result = service.ingest("annual_report.pdf")
        assert_eq(result.page_count, 2)
        assert_true(result.chunk_count >= 2)
        assert_eq(result.filename, "annual_report.pdf")


@test
def test_ingest_rejects_missing_file():
    with isolated_service() as (settings, service, pdf_dir):
        assert_raises(DocumentNotFound, service.ingest, "nope.pdf")


@test
def test_ingest_rejects_non_pdf():
    with isolated_service() as (settings, service, pdf_dir):
        (pdf_dir / "notes.txt").write_text("hello")
        assert_raises(UnreadableDocument, service.ingest, "notes.txt")


@test
def test_ingest_rejects_oversized_file():
    with isolated_service() as (settings, service, pdf_dir):
        tiny = Settings(data_dir=settings.data_dir, pdf_root=pdf_dir, max_pdf_bytes=10)
        assert_raises(DocumentTooLarge, SearchService(tiny).ingest, "annual_report.pdf")


@test
def test_ingest_rejects_pdf_without_extractable_text():
    with isolated_service() as (settings, service, pdf_dir):
        (pdf_dir / "blank.pdf").write_bytes(make_pdf_bytes([""]))
        assert_raises(UnreadableDocument, service.ingest, "blank.pdf")


@test
def test_ingest_rejects_unparseable_pdf():
    with isolated_service() as (settings, service, pdf_dir):
        (pdf_dir / "broken.pdf").write_bytes(b"%PDF-1.4\nthis is not a pdf at all")
        assert_raises(UnreadableDocument, service.ingest, "broken.pdf")


@test
def test_pagination_partitions_results_without_gaps_or_overlap():
    with isolated_service() as (settings, service, pdf_dir):
        service.ingest("annual_report.pdf")
        service.ingest("handbook.pdf")
        _, total = service.search("the", limit=1)
        collected = []
        for offset in range(0, total, 2):
            page, _ = service.search("the", limit=2, offset=offset)
            collected.extend(h.chunk_id for h in page)
        assert_eq(len(collected), total, "pages must cover every hit exactly once")
        assert_eq(len(set(collected)), total, "pages must not repeat a hit")


@test
def test_offset_beyond_total_returns_empty_page():
    with isolated_service() as (settings, service, pdf_dir):
        service.ingest("handbook.pdf")
        page, total = service.search("the", limit=10, offset=10_000)
        assert_eq(page, [])
        assert_true(total > 0, "total must still report the full match count")


print(f"{len(_TESTS)} unit tests registered")

31 unit tests registered


## 14. End-to-end tests against a live server

This is the part that actually proves the system works. A **real uvicorn server** starts on an
ephemeral port in a background thread, backed by an isolated temp corpus; tests drive it with
`requests` over real HTTP; the server is shut down cleanly afterwards.

This catches a whole class of bugs an in-process test client misses — serialisation, status
codes, middleware ordering, header propagation, and FastAPI's own request validation.

Startup uses a **readiness poll against `/health`**, not a fixed `sleep`. Sleeping is how you get
a suite that passes on your laptop and flakes in CI.

In [17]:
def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


@contextmanager
def live_server(settings: Settings, *, startup_timeout: float = 20.0):
    """Run the app under a real uvicorn server; yield its base URL."""
    port = _free_port()
    config = uvicorn.Config(
        build_app(settings), host="127.0.0.1", port=port, log_level="warning", access_log=False
    )
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True, name="e2e-uvicorn")
    thread.start()

    base_url = f"http://127.0.0.1:{port}"
    deadline = time.time() + startup_timeout
    while time.time() < deadline:
        if not thread.is_alive():
            raise RuntimeError("uvicorn thread died during startup")
        try:
            if requests.get(f"{base_url}/health", timeout=1).status_code == 200:
                break
        except requests.RequestException:
            time.sleep(0.05)
    else:
        server.should_exit = True
        raise TimeoutError(f"server did not become ready within {startup_timeout}s")

    try:
        yield base_url
    finally:
        server.should_exit = True
        thread.join(timeout=10)
        if thread.is_alive():
            log.warning("uvicorn thread did not shut down cleanly")


@contextmanager
def e2e_env():
    """Isolated corpus + sample PDFs + a running server."""
    with isolated_service() as (settings, service, pdf_dir):
        with live_server(settings) as base_url:
            yield base_url, pdf_dir

In [18]:
@test
def test_e2e_health_before_any_ingest():
    with e2e_env() as (url, _):
        response = requests.get(f"{url}/health", timeout=10)
        assert_eq(response.status_code, 200)
        assert_eq(response.json(), {"status": "healthy", "documents": 0, "chunks": 0})


@test
def test_e2e_every_response_carries_a_request_id():
    with e2e_env() as (url, _):
        response = requests.get(f"{url}/health", timeout=10)
        assert_true(response.headers.get("X-Request-ID"), "X-Request-ID header must be present")


@test
def test_e2e_client_supplied_request_id_is_echoed():
    with e2e_env() as (url, _):
        response = requests.get(f"{url}/health", headers={"X-Request-ID": "trace-me"}, timeout=10)
        assert_eq(response.headers["X-Request-ID"], "trace-me")


@test
def test_e2e_full_ingest_then_search_journey():
    """The primary happy path, exactly as a client would drive it."""
    with e2e_env() as (url, _):
        ingest = requests.post(f"{url}/documents", json={"pdf_path": "annual_report.pdf"}, timeout=30)
        assert_eq(ingest.status_code, 201, ingest.text)
        body = ingest.json()
        assert_eq(body["filename"], "annual_report.pdf")
        assert_eq(body["page_count"], 2)
        assert_true(body["chunk_count"] > 0)

        health = requests.get(f"{url}/health", timeout=10).json()
        assert_eq(health["documents"], 1)
        assert_eq(health["chunks"], body["chunk_count"])

        results = requests.get(f"{url}/search", params={"q": "revenue"}, timeout=10)
        assert_eq(results.status_code, 200)
        payload = results.json()
        assert_true(payload["total"] > 0, "expected hits for 'revenue'")
        first = payload["results"][0]
        assert_true("revenue" in first["text"].lower())
        assert_eq(first["filename"], "annual_report.pdf")
        assert_true(first["page_number"] >= 1)
        assert_true("revenue" in [t.lower() for t in first["matched_terms"]])


@test
def test_e2e_search_across_multiple_documents():
    with e2e_env() as (url, _):
        for name in SAMPLE_PAGES:
            assert_eq(requests.post(f"{url}/documents", json={"pdf_path": name}, timeout=30).status_code, 201)
        payload = requests.get(f"{url}/search", params={"q": "annual", "limit": 50}, timeout=10).json()
        filenames = {hit["filename"] for hit in payload["results"]}
        assert_eq(filenames, set(SAMPLE_PAGES), "'annual' appears in both sample documents")


@test
def test_e2e_mode_all_narrows_results():
    with e2e_env() as (url, _):
        for name in SAMPLE_PAGES:
            requests.post(f"{url}/documents", json={"pdf_path": name}, timeout=30)
        any_total = requests.get(f"{url}/search", params={"q": "annual expense", "mode": "any"}, timeout=10).json()["total"]
        all_total = requests.get(f"{url}/search", params={"q": "annual expense", "mode": "all"}, timeout=10).json()["total"]
        assert_true(all_total <= any_total, "AND must never return more than OR")
        assert_true(any_total > 0)


@test
def test_e2e_pagination_is_consistent():
    with e2e_env() as (url, _):
        requests.post(f"{url}/documents", json={"pdf_path": "annual_report.pdf"}, timeout=30)
        first = requests.get(f"{url}/search", params={"q": "the", "limit": 1, "offset": 0}, timeout=10).json()
        second = requests.get(f"{url}/search", params={"q": "the", "limit": 1, "offset": 1}, timeout=10).json()
        assert_eq(first["total"], second["total"], "total must not change between pages")
        if first["total"] > 1:
            assert_true(
                first["results"][0]["chunk_id"] != second["results"][0]["chunk_id"],
                "consecutive pages must return different hits",
            )


@test
def test_e2e_regex_injection_returns_200_not_500():
    """The headline fix: hostile queries that crash the original are handled cleanly."""
    with e2e_env() as (url, _):
        requests.post(f"{url}/documents", json={"pdf_path": "annual_report.pdf"}, timeout=30)
        for hostile in ["(", ")", "[", "a{2,", "*", "(a+)+$", "revenue)", ".*"]:
            response = requests.get(f"{url}/search", params={"q": hostile}, timeout=15)
            assert_true(
                response.status_code < 500,
                f"query {hostile!r} produced {response.status_code} — server error leaked",
            )


@test
def test_e2e_path_traversal_is_forbidden():
    with e2e_env() as (url, _):
        for attack in ("../../../../etc/passwd", "/etc/passwd", "../pdfs/../../../etc/hosts"):
            response = requests.post(f"{url}/documents", json={"pdf_path": attack}, timeout=15)
            assert_eq(response.status_code, 403, f"{attack!r} should be forbidden, got {response.text}")
            assert_eq(response.json()["code"], "unsafe_path")


@test
def test_e2e_missing_document_returns_404():
    with e2e_env() as (url, _):
        response = requests.post(f"{url}/documents", json={"pdf_path": "absent.pdf"}, timeout=15)
        assert_eq(response.status_code, 404)
        assert_eq(response.json()["code"], "document_not_found")


@test
def test_e2e_invalid_payload_returns_422():
    with e2e_env() as (url, _):
        assert_eq(requests.post(f"{url}/documents", json={}, timeout=10).status_code, 422)
        assert_eq(requests.post(f"{url}/documents", json={"pdf_path": ""}, timeout=10).status_code, 422)
        assert_eq(
            requests.post(f"{url}/documents",
                          json={"pdf_path": "annual_report.pdf", "chunk_size_words": 0},
                          timeout=10).status_code,
            422,
        )


@test
def test_e2e_query_parameter_bounds_are_enforced():
    with e2e_env() as (url, _):
        assert_eq(requests.get(f"{url}/search", params={"q": ""}, timeout=10).status_code, 422)
        assert_eq(requests.get(f"{url}/search", timeout=10).status_code, 422)
        assert_eq(requests.get(f"{url}/search", params={"q": "x", "limit": 0}, timeout=10).status_code, 422)
        assert_eq(requests.get(f"{url}/search", params={"q": "x", "limit": 9999}, timeout=10).status_code, 422)
        assert_eq(requests.get(f"{url}/search", params={"q": "x", "offset": -1}, timeout=10).status_code, 422)
        assert_eq(requests.get(f"{url}/search", params={"q": "x", "mode": "sideways"}, timeout=10).status_code, 422)


@test
def test_e2e_search_on_empty_corpus_returns_200():
    """The original raises ValueError from pd.concat here — a 500 on a fresh install."""
    with e2e_env() as (url, _):
        response = requests.get(f"{url}/search", params={"q": "anything"}, timeout=10)
        assert_eq(response.status_code, 200)
        assert_eq(response.json()["total"], 0)
        assert_eq(response.json()["results"], [])


@test
def test_e2e_document_lifecycle():
    with e2e_env() as (url, _):
        created = requests.post(f"{url}/documents", json={"pdf_path": "handbook.pdf"}, timeout=30).json()
        listing = requests.get(f"{url}/documents", timeout=10).json()
        assert_eq(len(listing), 1)
        assert_eq(listing[0]["document_id"], created["document_id"])

        deleted = requests.delete(f"{url}/documents/{created['document_id']}", timeout=10)
        assert_eq(deleted.status_code, 204)
        assert_eq(requests.get(f"{url}/documents", timeout=10).json(), [])
        assert_eq(requests.delete(f"{url}/documents/{created['document_id']}", timeout=10).status_code, 404)


@test
def test_e2e_custom_chunk_parameters_change_chunk_count():
    with e2e_env() as (url, _):
        coarse = requests.post(f"{url}/documents",
                               json={"pdf_path": "annual_report.pdf", "chunk_size_words": 200, "overlap_words": 0},
                               timeout=30).json()
        fine = requests.post(f"{url}/documents",
                             json={"pdf_path": "annual_report.pdf", "chunk_size_words": 10, "overlap_words": 2},
                             timeout=30).json()
        assert_true(fine["chunk_count"] > coarse["chunk_count"],
                    "smaller windows must produce more chunks")


@test
def test_e2e_concurrent_searches_are_consistent():
    """The corpus cache is shared across worker threads; verify it is not a data race."""
    with e2e_env() as (url, _):
        requests.post(f"{url}/documents", json={"pdf_path": "annual_report.pdf"}, timeout=30)
        totals, errors = [], []

        def query():
            try:
                totals.append(requests.get(f"{url}/search", params={"q": "revenue"}, timeout=15).json()["total"])
            except Exception as exc:
                errors.append(exc)

        threads = [threading.Thread(target=query) for _ in range(12)]
        for t in threads:
            t.start()
        for t in threads:
            t.join(timeout=30)

        assert_eq(errors, [], "concurrent queries must not error")
        assert_eq(len(set(totals)), 1, f"all concurrent queries must agree, got {set(totals)}")


@test
def test_e2e_openapi_schema_is_served():
    with e2e_env() as (url, _):
        schema = requests.get(f"{url}/openapi.json", timeout=10).json()
        for path in ("/health", "/search", "/documents"):
            assert_true(path in schema["paths"], f"{path} missing from the OpenAPI schema")


print(f"{len(_TESTS)} tests registered in total")

48 tests registered in total


## 15. Run the suite

Unit tests run in milliseconds. Each E2E test boots and tears down a real server, so the suite
takes a few seconds — that's the price of testing the actual transport.

In [19]:
failures = run_tests()
assert failures == 0, f"{failures} test(s) failed — see the report above"
print("\n✅ suite green")

running 48 tests
pass  test_windows_cover_all_words  (0 ms)
pass  test_windows_overlap_by_configured_amount  (0 ms)
pass  test_single_window_when_text_shorter_than_chunk_size  (0 ms)
pass  test_empty_input_yields_no_windows  (0 ms)
pass  test_window_terminates_and_does_not_duplicate_tail  (0 ms)
pass  test_words_are_never_split_mid_token  (7 ms)
pass  test_normalise_text_collapses_whitespace  (0 ms)
pass  test_traversal_outside_root_is_rejected  (5 ms)
pass  test_symlink_escaping_root_is_rejected  (4 ms)
pass  test_safe_stem_strips_dangerous_characters  (0 ms)
pass  test_regex_metacharacters_are_treated_as_literals  (1 ms)
pass  test_search_matches_whole_words_only  (0 ms)
pass  test_mode_all_requires_every_term  (0 ms)
pass  test_ranking_prefers_broader_term_coverage  (0 ms)
pass  test_ranking_is_deterministic  (0 ms)
pass  test_blank_query_is_rejected  (0 ms)
pass  test_too_many_terms_is_rejected  (0 ms)
pass  test_search_over_empty_corpus_returns_empty  (0 ms)


08:27:59 | INFO    | search_engine | stored 4 chunks for document_id=annual-report-34906302


pass  test_empty_corpus_does_not_raise  (3 ms)
pass  test_cache_is_invalidated_on_write  (14 ms)


08:27:59 | INFO    | search_engine | stored 4 chunks for document_id=annual-report-60509619
08:27:59 | INFO    | search_engine | stored 4 chunks for document_id=annual-report-60509619
08:27:59 | INFO    | search_engine | stored 2 chunks for document_id=handbook-61fa1cb4


pass  test_reingest_replaces_rather_than_duplicates  (21 ms)
pass  test_delete_removes_document  (8 ms)


...

08:27:59 | INFO    | search_engine | stored 4 chunks for document_id=annual-report-da366ae6
08:27:59 | INFO    | search_engine | stored 2 chunks for document_id=handbook-8f9d12c0


pass  test_ingest_rejects_unparseable_pdf  (6 ms)


08:27:59 | INFO    | search_engine | stored 2 chunks for document_id=handbook-5c52f839


pass  test_pagination_partitions_results_without_gaps_or_overlap  (20 ms)
pass  test_offset_beyond_total_returns_empty_page  (13 ms)
FAIL  test_e2e_health_before_any_ingest
FAIL  test_e2e_every_response_carries_a_request_id
FAIL  test_e2e_client_supplied_request_id_is_echoed
FAIL  test_e2e_full_ingest_then_search_journey
FAIL  test_e2e_search_across_multiple_documents
FAIL  test_e2e_mode_all_narrows_results
FAIL  test_e2e_pagination_is_consistent
FAIL  test_e2e_regex_injection_returns_200_not_500
FAIL  test_e2e_path_traversal_is_forbidden
FAIL  test_e2e_missing_document_returns_404
FAIL  test_e2e_invalid_payload_returns_422
FAIL  test_e2e_query_parameter_bounds_are_enforced
FAIL  test_e2e_search_on_empty_corpus_returns_200
FAIL  test_e2e_document_lifecycle
FAIL  test_e2e_custom_chunk_parameters_change_chunk_count
FAIL  test_e2e_concurrent_searches_are_consistent
FAIL  test_e2e_openapi_schema_is_served

--- test_e2e_health_before_any_ingest ---
Traceback (most recent call last):
  File 

AssertionError: 17 test(s) failed — see the report above

## 16. Try it interactively

A live server against the real `Day_1` directory, so you can drive the API by hand.

In [ ]:
playground_root = Path(tempfile.mkdtemp(prefix="playground-"))
pdf_dir = playground_root / "pdfs"
pdf_dir.mkdir()
for name, pages in SAMPLE_PAGES.items():
    (pdf_dir / name).write_bytes(make_pdf_bytes(pages))

playground_settings = Settings(
    data_dir=playground_root / "csv_files", pdf_root=pdf_dir,
    chunk_size_words=40, overlap_words=10,
).ensure_dirs()

with live_server(playground_settings) as url:
    print(f"server: {url}   docs: {url}/docs\n")

    for name in SAMPLE_PAGES:
        created = requests.post(f"{url}/documents", json={"pdf_path": name}, timeout=30).json()
        print(f"ingested {created['filename']:20} {created['chunk_count']} chunks "
              f"across {created['page_count']} page(s)")

    for query, mode in [("revenue", "any"), ("annual leave", "all"), ("revenue growth", "any")]:
        payload = requests.get(f"{url}/search", params={"q": query, "mode": mode, "limit": 3}, timeout=10).json()
        print(f"\nq={query!r} mode={mode} → {payload['total']} hit(s)")
        for hit in payload["results"]:
            snippet = hit["text"][:88] + ("…" if len(hit["text"]) > 88 else "")
            print(f"   [{hit['score']:>5}] {hit['filename']} p{hit['page_number']}c{hit['chunk_number']}: {snippet}")

shutil.rmtree(playground_root, ignore_errors=True)

## 17. Running it as a service

To run this outside the notebook, the equivalent of `python search_engine.py`:

```python
# search_engine_v2.py
settings = Settings.from_env()
app = build_app(settings)

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=9321)
```

```bash
SEARCH_DATA_DIR=./csv_files SEARCH_PDF_ROOT=./pdfs python search_engine_v2.py
# or, for multiple workers:
uvicorn search_engine_v2:app --host 127.0.0.1 --port 9321 --workers 4
```

Note `127.0.0.1`, not the original's `0.0.0.0` — binding to all interfaces exposes the service to
your whole network. Bind to loopback in development and put a reverse proxy in front in
production.

### Adapting `search_ui.py`

The Streamlit client calls `POST /search_chunks?search_string=...` and does
`pd.read_json(response.json())`. Against this API it becomes:

```python
response = requests.get(f"{API_URL}/search", params={"q": search_input, "limit": 50})
response.raise_for_status()
results = pd.DataFrame(response.json()["results"])
```

`search_ui.py` also renders chunk text with `unsafe_allow_html=True`, which makes any HTML inside
a PDF executable in the browser — an XSS sink. Use `st.text()` / `st.markdown()` without the flag,
or escape with `html.escape()` first.

### What a production deployment would add

| Concern | Approach |
|---------|----------|
| Relevance | CSV + regex is fine to ~10⁴ chunks; beyond that use SQLite FTS5, then a vector store (Week 4+ covers this) |
| Ingestion latency | Large PDFs block the request — move to a task queue and return `202 Accepted` with a job id |
| Auth | No authentication here; add an API key or OAuth before exposing it |
| Rate limiting | Per-client quotas so one caller cannot saturate the service |
| Observability | Prometheus metrics on latency/error rate; structured JSON logs |
| Persistence | CSV files don't survive an ephemeral container — mount a volume or use object storage |
| Concurrency | The write lock is per-process; multi-worker deployments need a shared store |

### Exercises

1. **Phrase search** — support `"annual leave"` as a quoted phrase that must match contiguously.
   Add a test proving it does not match `"annual"` and `"leave"` far apart.
2. **Snippet highlighting** — return a windowed excerpt around each match instead of the whole
   chunk. Test the window clamps correctly at chunk boundaries.
3. **Upload endpoint** — accept `UploadFile` instead of a server-side path, which removes the
   traversal surface entirely. Test the size limit is enforced during streaming, not after.
4. **Stemming** — make `growing` match `growth`. Test that precision doesn't collapse.
5. **Swap the backend** — reimplement `ChunkStore` on SQLite FTS5. Every test above should pass
   unchanged; that's the payoff of testing behaviour rather than implementation.